# تدريب موديل كشف المخالفات — 3 فئات
**الفئات:** `illegal_dumping` (نفايات مرمية) · `road_damage` (حفر) · `abandoned_vehicle` (سيارات مهجورة)

**الطريقة:** YOLO26n جاهز (Pretrained) ← Fine-tuning ← تصدير NCNN للتشغيل على Raspberry Pi 5.

**قبل البدء:** `Runtime > Change runtime type > T4 GPU` ثم شغّل الخلايا بالترتيب.

> لو انقطعت الجلسة: شغّل الخلايا من 1 إلى 6 مرة ثانية. التنزيل ما يتعاد لأن الداتا محفوظة في Drive، والتدريب يكمل من آخر نقطة.

In [ ]:
# 1) ربط Google Drive وتجهيز المسارات
# كل شي مهم (الداتا الخام، نقاط التدريب، النتائج) ينحفظ في Drive عشان ما يضيع لو انقطعت الجلسة
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/waste_project'
RAW_DIR    = f'{DRIVE_ROOT}/raw'           # الداتا الخام بعد التنزيل
RUN_DIR    = f'{DRIVE_ROOT}/runs'          # نقاط التدريب
OUT_DIR    = f'{DRIVE_ROOT}/deliverables'  # الملفات النهائية للتقرير
MERGED     = '/content/merged'             # الداتا المدمجة على قرص Colab (أسرع في التدريب)
for d in (RAW_DIR, RUN_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
!nvidia-smi -L

In [ ]:
# 2) تثبيت المكتبات
!pip install -q -U ultralytics roboflow
import ultralytics
ultralytics.checks()

## تنزيل الداتا من Roboflow (حساب مجاني)
المفتاح من: app.roboflow.com ← Settings ← API Keys.
الأفضل تحطه في Colab Secrets (أيقونة المفتاح يسار) باسم `ROBOFLOW_API_KEY`، وإلا بيطلبه منك.

- `None` في رقم النسخة = آخر نسخة منشورة (رقمها ينطبع).
- **النفايات والحفر إلزامية:** لو فشل تنزيلها يوقف.
- **السيارات المهجورة اختيارية:** لو فشل تنزيلها يكمل بفئتين وينبهك.

In [ ]:
# 3) تنزيل الداتا (مرة وحدة، وبعدها تنقرأ من Drive)
from roboflow import Roboflow
from getpass import getpass
try:
    from google.colab import userdata
    API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    API_KEY = None
if not API_KEY:
    API_KEY = getpass('Roboflow API key: ')
rf = Roboflow(api_key=API_KEY)

# (اسم مختصر, workspace, project, version, إلزامي؟)
SOURCES = [
    ('smart', 'smartathon-n3kfm', 'visual-pollution-iojfm',      None, True),   # شوارع السعودية: GARBAGE + POTHOLES
    ('dump',  'bill-lhxqf',       'illegal-dumping-detection-2', 1,    True),   # نفايات مرمية (نحذف person)
    ('dead',  'yin',              'deadvehicle',                 None, False),  # سيارات مهجورة
]

DATASETS = []
for short, ws, proj, ver, required in SOURCES:
    loc = f'{RAW_DIR}/{short}'
    if os.path.exists(f'{loc}/data.yaml'):
        print(f'[{short}] موجودة في Drive، ما بنزلها مرة ثانية')
        DATASETS.append((short, loc))
        continue
    try:
        project = rf.workspace(ws).project(proj)
        if ver is None:
            ver = max(int(v['id'].split('/')[-1]) for v in project.get_version_information())
        print(f'[{short}] تنزيل {ws}/{proj} نسخة {ver}')
        project.version(ver).download('yolov8', location=loc)
        DATASETS.append((short, loc))
    except Exception as e:
        if required:
            raise RuntimeError(f'فشل تنزيل {short} وهي إلزامية: {e}')
        print(f'تنبيه: فشل تنزيل {short}: {e}\nبنكمل بدونها (فئتين بس)')

# نفتح data.yaml حق كل داتا ونتأكد من أسماء الفئات قبل الدمج
for short, loc in DATASETS:
    print(f'\n===== {short}: data.yaml =====')
    print(open(f'{loc}/data.yaml').read())

## 4) سكربت الدمج (نفس ملف `merge_datasets.py`)

In [ ]:
%%writefile /content/merge_datasets.py
# merge_datasets.py
# يدمج عدة داتا سيتات بصيغة YOLO في داتا وحدة بفئات موحدة:
#   0 = illegal_dumping   1 = road_damage   2 = abandoned_vehicle
#
# - أسماء الفئات تنقرأ من data.yaml حق كل داتا، والربط بالاسم بالضبط (مو بالرقم ولا بجزء من الكلمة)
#   لو اسم مطلوب ما لقيناه، السكربت يوقف ويطبع الأسماء الموجودة بدل ما يدمج غلط
# - الفئات غير المطلوبة (person, graffiti, ...) تنحذف، والصور اللي ما بقى فيها شي تتخطى
# - الصور المكررة بنفس المحتوى بين المصادر تنحذف (عشان ما يصير تسريب بين train و test)
# - الـ polygons (لو الداتا segmentation) تتحول لمربعات
# - لو مصدر السيارات المهجورة مو موجود، الداتا تطلع بفئتين بس (0 و 1)
#
# الاستخدام:
#   python merge_datasets.py output_dir smart=path1 dump=path2 dead=path3

import hashlib
import json
import os
import shutil
import sys
from collections import defaultdict

import yaml

ALL_CLASSES = {0: "illegal_dumping", 1: "road_damage", 2: "abandoned_vehicle"}

# جدول الربط لكل مصدر: رقم الفئة الموحد <- الأسماء المقبولة (بحروف صغيرة، مطابقة كاملة)
MAPPING = {
    "smart": {0: ["garbage"], 1: ["potholes", "pothole"]},
    "dump": {0: ["garbage"]},
    "dead": {2: ["deadvehicle", "dead vehicle", "dead_vehicle", "dead-vehicle",
                 "abandoned vehicle", "abandoned_vehicle", "abandoned-vehicle"]},
}
# مصادر لو فيها فئة وحدة بس، تنربط بهالفئة تلقائياً حتى لو اسمها مختلف
SINGLE_CLASS_DEFAULT = {"dead": 2}

SPLITS = {"train": "train", "valid": "valid", "val": "valid", "test": "test"}
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def read_names(dataset_dir):
    """يقرأ أسماء الفئات من data.yaml (قائمة أو قاموس)"""
    path = os.path.join(dataset_dir, "data.yaml")
    if not os.path.exists(path):
        sys.exit(f"[خطأ] ما لقيت data.yaml في {dataset_dir}")
    with open(path, encoding="utf-8") as f:
        names = yaml.safe_load(f).get("names")
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names, key=int)]
    if not names:
        sys.exit(f"[خطأ] data.yaml في {dataset_dir} ما فيه names")
    return [str(n) for n in names]


def build_mapping(short, names):
    """يطلع جدول تحويل: رقم الفئة القديم -> الرقم الموحد. الفئات اللي مو فيه تنحذف"""
    if short not in MAPPING:
        sys.exit(f"[خطأ] المصدر '{short}' مو معروف. المصادر المعروفة: {list(MAPPING)}")
    lower = [n.strip().lower() for n in names]
    mapping = {}
    if len(names) == 1 and short in SINGLE_CLASS_DEFAULT:
        mapping[0] = SINGLE_CLASS_DEFAULT[short]
    else:
        for new_id, accepted in MAPPING[short].items():
            found = [i for i, n in enumerate(lower) if n in accepted]
            if not found:
                sys.exit(f"[خطأ] في '{short}' ما لقيت فئة {ALL_CLASSES[new_id]} "
                         f"(الأسماء المقبولة: {accepted}).\n"
                         f"       الفئات الموجودة: {names}\n"
                         f"       أضف الاسم الصحيح في MAPPING داخل merge_datasets.py")
            for i in found:
                mapping[i] = new_id

    print(f"\n[{short}] الفئات الأصلية والتحويل:")
    for i, n in enumerate(names):
        print(f"   {i}: {n:<25} -> {ALL_CLASSES[mapping[i]] if i in mapping else 'حذف'}")
    return mapping


def find_splits(dataset_dir):
    """يدور على مجلدات train/valid/test اللي فيها images"""
    splits = {}
    for d in sorted(os.listdir(dataset_dir)):
        full = os.path.join(dataset_dir, d)
        if d.lower() in SPLITS and os.path.isdir(os.path.join(full, "images")):
            splits[SPLITS[d.lower()]] = full
    if not splits:
        sys.exit(f"[خطأ] ما لقيت مجلدات train/valid/test في {dataset_dir}")
    return splits


def convert_line(line, mapping):
    """يحوّل سطر label واحد. يرجع (الفئة, السطر) أو None لو الفئة محذوفة أو السطر تالف"""
    parts = line.split()
    if len(parts) < 5:
        return None
    try:
        cls = int(float(parts[0]))
        vals = [float(v) for v in parts[1:]]
    except ValueError:
        return None
    if cls not in mapping:
        return None
    if len(vals) == 4:
        x, y, w, h = vals
    elif len(vals) >= 6 and len(vals) % 2 == 0:  # polygon: x1 y1 x2 y2 ...
        xs, ys = vals[0::2], vals[1::2]
        x, y = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
        w, h = max(xs) - min(xs), max(ys) - min(ys)
    else:
        return None
    # نقص القيم داخل حدود الصورة ونتخطى المربعات الصفرية
    x, y, w, h = (min(max(v, 0.0), 1.0) for v in (x, y, w, h))
    if w <= 0 or h <= 0:
        return None
    new = mapping[cls]
    return new, f"{new} {x:.6f} {y:.6f} {w:.6f} {h:.6f}"


def file_hash(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def merge(sources, output):
    """sources: قائمة (اسم_مختصر, مسار_المجلد). يرجع الإحصائية"""
    # نقرأ كل الجداول أول، عشان لو فيه خطأ يوقف قبل ما ينسخ أي شي
    mappings = {short: build_mapping(short, read_names(path)) for short, path in sources}
    used = sorted({c for m in mappings.values() for c in m.values()})
    if used != list(range(len(used))):
        sys.exit(f"[خطأ] أرقام الفئات المستخدمة لازم تكون متتالية من 0، والموجود: {used}")
    final = {c: ALL_CLASSES[c] for c in used}

    if os.path.exists(output):
        shutil.rmtree(output)
    for s in ("train", "valid", "test"):
        os.makedirs(os.path.join(output, s, "images"))
        os.makedirs(os.path.join(output, s, "labels"))

    # stats[split][class] = {"images": n, "boxes": n}
    stats = {s: {n: {"images": 0, "boxes": 0} for n in final.values()} for s in ("train", "valid", "test")}
    images = defaultdict(int)
    skipped = {short: {"empty": 0, "duplicate": 0, "no_label": 0} for short, _ in sources}
    seen = set()

    for short, path in sources:
        mapping = mappings[short]
        for split, split_dir in find_splits(path).items():
            img_dir = os.path.join(split_dir, "images")
            for img_name in sorted(os.listdir(img_dir)):
                base, ext = os.path.splitext(img_name)
                if ext.lower() not in IMG_EXTS:
                    continue
                lbl = os.path.join(split_dir, "labels", base + ".txt")
                if not os.path.exists(lbl):
                    skipped[short]["no_label"] += 1
                    continue
                with open(lbl, encoding="utf-8", errors="ignore") as f:
                    lines = [r for r in (convert_line(l, mapping) for l in f) if r]
                if not lines:  # ما بقى فيها شي من فئاتنا
                    skipped[short]["empty"] += 1
                    continue
                img_path = os.path.join(img_dir, img_name)
                digest = file_hash(img_path)
                if digest in seen:  # نفس الصورة موجودة من قبل
                    skipped[short]["duplicate"] += 1
                    continue
                seen.add(digest)

                new_base = f"{short}_{base}"
                shutil.copy(img_path, os.path.join(output, split, "images", new_base + ext.lower()))
                with open(os.path.join(output, split, "labels", new_base + ".txt"), "w", encoding="utf-8") as f:
                    f.write("\n".join(l for _, l in lines) + "\n")

                images[split] += 1
                for c in {c for c, _ in lines}:
                    stats[split][final[c]]["images"] += 1
                for c, _ in lines:
                    stats[split][final[c]]["boxes"] += 1

    data = {
        "path": os.path.abspath(output),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": len(final),
        "names": final,
    }
    with open(os.path.join(output, "data.yaml"), "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)

    print("\n===== الإحصائية (عدد الصور / عدد المربعات لكل فئة) =====")
    print(f"{'split':<7}{'class':<20}{'images':>8}{'boxes':>8}")
    for s in ("train", "valid", "test"):
        for n in final.values():
            print(f"{s:<7}{n:<20}{stats[s][n]['images']:>8}{stats[s][n]['boxes']:>8}")
        print(f"{s:<7}{'(كل الصور)':<20}{images[s]:>8}")
    print("-" * 43)
    for short, d in skipped.items():
        print(f"[{short}] تخطينا: فاضية بعد الحذف {d['empty']} | مكررة {d['duplicate']} | بدون label {d['no_label']}")
    for n in final.values():
        if stats["train"][n]["boxes"] == 0:
            print(f"تنبيه: الفئة {n} ما لها أي صورة في train")
    if images["test"] == 0:
        print("تنبيه: ما فيه صور test، التقييم بيكون على valid")
    if 2 not in final:
        print("تنبيه: ما فيه داتا سيارات مهجورة، الموديل بيتدرب على فئتين بس")

    summary = {"classes": final, "per_split": stats, "images_per_split": dict(images), "skipped": skipped}
    with open(os.path.join(output, "merge_stats.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print(f"\nتم. data.yaml: {os.path.join(output, 'data.yaml')}")
    return summary


if __name__ == "__main__":
    if len(sys.argv) < 3:
        sys.exit("الاستخدام: python merge_datasets.py output_dir smart=path1 dump=path2 [dead=path3]")
    srcs = [tuple(a.split("=", 1)) for a in sys.argv[2:]]
    merge(srcs, sys.argv[1])

In [ ]:
# 5) الدمج وتوحيد الفئات + الإحصائية
# راجع جدول التحويل: كل فئة لازم تروح لمكانها الصح. لو اسم ناقص يوقف ويقولك وش تضيف
args = ' '.join(f'{s}="{p}"' for s, p in DATASETS)
!python /content/merge_datasets.py {MERGED} {args}
!cp {MERGED}/merge_stats.json "{OUT_DIR}/"

## التدريب
- يبدأ من `yolo26n.pt` الجاهز (Fine-tuning، مو من الصفر).
- `epochs=50` مع `patience=20` (يوقف بدري لو ما تحسن 20 epoch). لو النتائج طيبة والوقت يسمح، زدها لـ 100 **قبل أول تشغيل**.
- `last.pt` ينحفظ في Drive كل epoch. لو انقطعت الجلسة، الخلية تكمل منه تلقائياً.
- لو التدريب خلص من قبل، الخلية تتخطاه (ما تبدأ تدريب جديد بالغلط).

In [ ]:
# 6) التدريب (يكمل تلقائياً من آخر نقطة لو انقطع)
import torch
from ultralytics import YOLO
EPOCHS = 50
LAST = f'{RUN_DIR}/waste/weights/last.pt'

if os.path.exists(LAST):
    ckpt = torch.load(LAST, map_location='cpu', weights_only=False)
    if ckpt.get('optimizer') is None:
        # Ultralytics يشيل حالة الـ optimizer من last.pt لما يخلص التدريب كامل
        # لو سوينا resume هنا بيبدأ تدريب جديد على coco8، فنتخطى
        print('التدريب مكتمل مسبقاً. كمّل للخلية اللي بعدها.')
    else:
        print(f"تكملة التدريب من epoch {ckpt['epoch'] + 1}:", LAST)
        YOLO(LAST).train(resume=True)
    del ckpt
else:
    model = YOLO('yolo26n.pt')  # الموديل المدرَّب مسبقاً
    model.train(
        data=f'{MERGED}/data.yaml',
        epochs=EPOCHS,
        imgsz=640,
        batch=16,
        patience=20,   # يوقف بدري لو ما تحسن
        workers=2,
        project=RUN_DIR,
        name='waste',
        exist_ok=True,
        seed=0,
        plots=True,
    )

## التقييم: mAP50 و mAP50-95 لكل فئة (على valid و test)

In [ ]:
# 7) جدول النتائج لكل فئة
import pandas as pd
W_DIR = f'{RUN_DIR}/waste'
BEST = f'{W_DIR}/weights/best.pt'
best = YOLO(BEST)
has_test = len(os.listdir(f'{MERGED}/test/images')) > 0
EVAL_SPLITS = ['val', 'test'] if has_test else ['val']
MAIN = EVAL_SPLITS[-1]  # التقييم الأساسي: test لو موجود

rows = []
for split in EVAL_SPLITS:
    m = best.val(data=f'{MERGED}/data.yaml', split=split, imgsz=640, batch=16,
                 project=W_DIR, name=f'eval_{split}', exist_ok=True, plots=True)
    for i, c in enumerate(m.box.ap_class_index):
        rows.append({'split': split, 'class': best.names[int(c)],
                     'precision': m.box.p[i], 'recall': m.box.r[i],
                     'mAP50': m.box.ap50[i], 'mAP50-95': m.box.ap[i]})
    rows.append({'split': split, 'class': 'all', 'precision': m.box.mp, 'recall': m.box.mr,
                 'mAP50': m.box.map50, 'mAP50-95': m.box.map})

results = pd.DataFrame(rows).round(3)
results.to_csv(f'{OUT_DIR}/results_table.csv', index=False)
results

In [ ]:
# 8) منحنيات التدريب و Confusion Matrix
from IPython.display import Image, display
for f in [f'{W_DIR}/results.png',
          f'{W_DIR}/eval_{MAIN}/confusion_matrix_normalized.png',
          f'{W_DIR}/eval_{MAIN}/BoxPR_curve.png']:
    if os.path.exists(f):
        print(f); display(Image(f, width=900))
    else:
        print('مو موجود:', f)

## تحليل الأخطاء على صور ما شافها الموديل وقت التدريب

In [ ]:
# 9) لكل صورة: كم مربع صح (TP)، كم زايد (FP)، كم فاته (FN)، وكم فئته غلط
import glob

def load_gt(lbl_path, w, h):
    # يقرأ الـ labels الحقيقية ويحولها لإحداثيات بكسل
    out = []
    if os.path.exists(lbl_path):
        for line in open(lbl_path):
            c, x, y, bw, bh = map(float, line.split()[:5])
            out.append((int(c), (x-bw/2)*w, (y-bh/2)*h, (x+bw/2)*w, (y+bh/2)*h))
    return out

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

def label_of(img_path):
    return img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'

CONF, IOU_T = 0.4, 0.5   # نفس الـ conf الافتراضي في pi_detect.py
eval_dir = 'test' if MAIN == 'test' else 'valid'
eval_imgs = sorted(glob.glob(f'{MERGED}/{eval_dir}/images/*'))
per_image = []
for r in best.predict(eval_imgs, imgsz=640, conf=CONF, stream=True, verbose=False):
    h, w = r.orig_shape
    gt = load_gt(label_of(r.path), w, h)
    preds = [(int(c), *b) for c, b in zip(r.boxes.cls.tolist(), r.boxes.xyxy.tolist())]
    used, tp, wrong_cls = set(), 0, 0
    for p in preds:
        best_j, best_iou = -1, 0
        for j, g in enumerate(gt):
            if j in used: continue
            v = iou(p[1:], g[1:])
            if v > best_iou: best_j, best_iou = j, v
        if best_iou >= IOU_T:
            used.add(best_j)
            if gt[best_j][0] == p[0]: tp += 1
            else: wrong_cls += 1
    per_image.append({'image': r.path, 'source': os.path.basename(r.path).split('_')[0],
                      'gt': len(gt), 'pred': len(preds), 'tp': tp,
                      'fp': len(preds) - tp - wrong_cls, 'fn': len(gt) - len(used),
                      'wrong_class': wrong_cls})

err = pd.DataFrame(per_image)
err['errors'] = err['fp'] + err['fn'] + err['wrong_class']
err.to_csv(f'{OUT_DIR}/per_image_errors.csv', index=False)
print(f'على {eval_dir}:', err[['gt','pred','tp','fp','fn','wrong_class']].sum().to_dict())
print('\nحسب المصدر:'); print(err.groupby('source')[['gt','tp','fp','fn','wrong_class']].sum())

In [ ]:
# 10) 6 صور نجاح + 6 صور أخطاء (الأخضر = الحقيقة، الملون = توقع الموديل)
import cv2, matplotlib.pyplot as plt

def draw(path):
    r = best.predict(path, imgsz=640, conf=CONF, verbose=False)[0]
    img = r.plot(line_width=2)
    h, w = r.orig_shape
    for c, x0, y0, x1, y1 in load_gt(label_of(path), w, h):
        cv2.rectangle(img, (int(x0), int(y0)), (int(x1), int(y1)), (0, 255, 0), 2)
    return img[..., ::-1]

def grid(paths, title, fname):
    if not paths:
        print('ما فيه صور لـ:', title); return
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, p in zip(axes.flat, paths):
        ax.imshow(draw(p)); ax.set_title(os.path.basename(p)[:40], fontsize=9)
    for ax in axes.flat: ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); fig.savefig(f'{OUT_DIR}/{fname}', dpi=110); plt.show()

ok  = err[err['errors'] == 0].sample(min(6, int((err['errors'] == 0).sum())), random_state=0)
bad = err[err['errors'] > 0].sort_values('errors', ascending=False).head(6)
grid(ok['image'].tolist(),  'Correct detections', 'examples_success.png')
grid(bad['image'].tolist(), 'Worst errors',       'examples_errors.png')
bad[['image','gt','pred','tp','fp','fn','wrong_class']]

## (اختياري) جرّب على صورك الخاصة

In [ ]:
# 11) ارفع صور من جوالك أو من الشارع
from google.colab import files
up = files.upload()
for name in up:
    r = best.predict(name, imgsz=640, conf=CONF, verbose=False)[0]
    plt.figure(figsize=(10, 7)); plt.imshow(r.plot()[..., ::-1]); plt.axis('off'); plt.title(name); plt.show()
    print(name, [(best.names[int(c)], round(float(s), 2)) for c, s in zip(r.boxes.cls, r.boxes.conf)])

## التصدير للـ Raspberry Pi 5
NCNN هي الصيغة الأسرع على معالجات ARM مثل الرازبيري. الملف `best_ncnn_model.zip` هو اللي يحتاجه `pi_detect.py`.

In [ ]:
# 12) تصدير NCNN + تجميع الملفات النهائية في Drive وتنزيل الموديل
import shutil
from google.colab import files
out = best.export(format='ncnn', imgsz=640)
print('المجلد:', out)
shutil.make_archive('/content/best_ncnn_model', 'zip', os.path.dirname(out), os.path.basename(out))
shutil.copy('/content/best_ncnn_model.zip', OUT_DIR)
shutil.copy(BEST, f'{OUT_DIR}/best.pt')
for f in ['results.png', 'results.csv', 'args.yaml']:
    if os.path.exists(f'{W_DIR}/{f}'): shutil.copy(f'{W_DIR}/{f}', OUT_DIR)
for f in glob.glob(f'{W_DIR}/eval_{MAIN}/*.png'):
    shutil.copy(f, f'{OUT_DIR}/{MAIN}_{os.path.basename(f)}')
print('في Drive:', sorted(os.listdir(OUT_DIR)))
files.download('/content/best_ncnn_model.zip')